# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced using their unique `@id`.

### Dataset Source
FAIR² Croissant schema: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Explore available record sets, their IDs, fields, and columns in the dataset.

In [ ]:
# List all available record sets by @id and name
print("Available record sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"  RecordSet @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For each record set, print field @ids and names
for rs in record_sets:
    print(f"\nFields in RecordSet {rs['@id']}: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id', None)
            field_name = field.get('name', 'N/A')
        else:
            field_id = field
            field_name = 'N/A'
        print(f"    Field @id: {field_id}, name: {field_name}")

## 3. Data Extraction
Load records from each record set into Pandas DataFrames using the record set `@id`s found above.
All entities (record sets, fields, columns) are always referenced by their `@id`.

In [ ]:
# Extract data from each detected record set (@id)
from collections import OrderedDict
dataframes = {}

# List of all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"\nRecord set @ids: {record_set_ids}")

for record_set_id in record_set_ids:
    # Records generator, may be empty if no records exist
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet {record_set_id} with shape {df.shape}")
    else:
        print(f"No records extracted for RecordSet {record_set_id}.")

# List columns for first loaded dataframe, if available
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in first extracted RecordSet ({first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Process the dataset: filtering, normalization, categorization, and groupings.
All fields used here are referenced by their `@id`s.

In [ ]:
# Choose a numeric field and a record set for processing (edit these values using printed @ids above)
# Example: use first record set and a numeric field within it

if dataframes:
    # Pick the first DataFrame and try to find a numeric column for demo purposes
    record_set_id = first_rs_id
    df = dataframes[record_set_id]
    # Try to auto-detect numeric columns
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using RecordSet @id: {record_set_id}")
        print(f"Numeric field @id: {numeric_field_id}")
        # Filtering on this column (arbitrary threshold)
        threshold = df[numeric_field_id].quantile(0.75) # 75th percentile as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        
        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try to find a group field (categorical)
        group_candidates = df.select_dtypes(include=['object']).columns.tolist()
        if group_candidates:
            group_field = group_candidates[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
                print(f"\nGrouped data by {group_field}:")
                display(grouped_df.head())
        else:
            print("No categorical grouping field detected.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No dataframes could be loaded from the dataset record sets.")

## 5. Visualization
Visualize distributions and basic relationships between fields.

> _If available:_ Histogram of the numeric field; boxplot grouped by the first categorical field (using their `@id`s).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    # Histogram of numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Boxplot by the first categorical field (if possible)
    if group_candidates:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_candidates[0]], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_candidates[0]}")
        plt.xlabel(group_candidates[0])
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load FAIR² dataset metadata and records via the Croissant schema using `mlcroissant`
- Reference all dataset structures by their `@id`
- Explore available record sets and their fields programmatically
- Extract records into `pandas` DataFrames by record set `@id`
- Conduct basic exploratory data analysis and visualizations by referencing core variables by `@id`

_You can adapt this template for any Croissant dataset by adjusting record set/field `@id`s as displayed in the Data Overview. Further steps may include cleaning and modeling using the loaded DataFrames._